In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.electiva.datos_csv;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.electiva.customers
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/customers.csv', format => 'csv', header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.employee
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/employees.csv',format => 'csv',header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.orders
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/orders.csv',format => 'csv',header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.product_categories
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/productcategories.csv',format => 'csv',header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/productsubcategories.csv',format => 'csv',header => true, inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.products
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/products.csv',format => 'csv',header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.vendor
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/vendors.csv',format => 'csv',header => true,
  inferSchema => true);

CREATE OR REPLACE TABLE workspace.electiva.vendor_product
USING DELTA
AS SELECT * FROM read_files('/Volumes/workspace/electiva/datos_csv/vendorproduct.csv',format => 'csv',header => true,
  inferSchema => true);

In [0]:
%sql
SELECT table_name
FROM workspace.information_schema.tables
WHERE table_schema = 'electiva'
AND table_name IN (
    'customers',
    'employee',
    'orders',
    'product_categories',
    'product_sub_categories',
    'products',
    'vendor',
    'vendor_product'
)
ORDER BY table_name;

**Punto 1. Construcción del nombre de usuario**

In [0]:
%sql
SELECT CustomerID AS customerid, CONCAT(FirstName, ' ', LastName) AS fullname, CONCAT(CAST(CustomerID AS STRING),'-',
        UPPER(SUBSTRING(FirstName, 1, 2)), '-', LOWER(SUBSTRING(LastName, 1, 2)), '-AW') AS user_name
FROM workspace.electiva.customers;

En este punto se realizó la construcción de un nombre de usuario único para cada cliente, utilizando su identificador, las dos primeras letras de su nombre en mayúsculas y las dos primeras letras de su apellido en minúsculas, seguido de la terminación AW. Este procedimiento permitió transformar los datos personales de los clientes en una estructura estandarizada, facilitando su identificación y consulta dentro del sistema.

**Punto 2. Crear la tabla page_user**

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.page_user (customerid INT,user_name STRING)
USING DELTA;

INSERT INTO workspace.electiva.page_user
SELECT CustomerID AS customerid, CONCAT(CAST(CustomerID AS STRING), '-', UPPER(SUBSTRING(FirstName, 1, 2)), '-',
LOWER(SUBSTRING(LastName, 1, 2)), '-AW') AS user_name
FROM workspace.electiva.customers;

In [0]:
%sql
SELECT * FROM workspace.electiva.page_user;

Se creó la tabla page_user para almacenar directamente el identificador de cada cliente y su respectivo nombre de usuario. Se insertaron correctamente 635 registros, lo que permitió disponer de esta información en una tabla independiente y facilitar su utilización en consultas posteriores, especialmente en la relación entre clientes y órdenes de compra.

**Punto 3. Descuentos por categoría – agosto de 2011**

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.electiva.orders_august_2011 AS

SELECT
    p.ProductName AS productname,
    p.ModelName AS modelname,
    pu.user_name,
    o.SalesOrderID AS salesorderid,
    pc.CategoryName AS categoryname,
    o.OrderDate AS orderdate,
    o.LineTotal AS linetotal,

    CASE
        WHEN pc.CategoryName = 'Bikes' THEN 0.10
        WHEN pc.CategoryName = 'Clothing' THEN 0.15
        WHEN pc.CategoryName = 'Components' THEN 0.08
        WHEN pc.CategoryName = 'Accessories' THEN 0.20
        ELSE 0
    END AS `descuento_mitad_año`,

    o.LineTotal - (
        o.LineTotal *
        CASE
            WHEN pc.CategoryName = 'Bikes' THEN 0.10
            WHEN pc.CategoryName = 'Clothing' THEN 0.15
            WHEN pc.CategoryName = 'Components' THEN 0.08
            WHEN pc.CategoryName = 'Accessories' THEN 0.20
            ELSE 0
        END
    ) AS total_descuentos_aplicados

FROM workspace.electiva.orders o

INNER JOIN workspace.electiva.products p
    ON o.ProductID = p.ProductID

INNER JOIN workspace.electiva.product_sub_categories psc
    ON p.SubCategoryID = psc.SubCategoryID

INNER JOIN workspace.electiva.product_categories pc
    ON psc.CategoryID = pc.CategoryID

INNER JOIN workspace.electiva.page_user pu
    ON o.CustomerID = pu.customerid

WHERE YEAR(o.OrderDate) = 2011
  AND MONTH(o.OrderDate) = 8;

In [0]:
%sql
SELECT *
FROM workspace.electiva.orders_august_2011
LIMIT 100;

Se desarrolló una vista que reúne las órdenes realizadas durante agosto de 2011, relacionando la información de los productos, las categorías, los clientes y sus respectivas órdenes. Se aplicaron los porcentajes de descuento establecidos para cada categoría y se calculó el valor final después del descuento.

El resultado permitió identificar cómo varía el valor de las compras según la categoría del producto, facilitando el análisis de una estrategia comercial basada en descuentos de temporada. La consulta obtuvo 1.092 registros correspondientes al periodo solicitado.

**Punto 4. Clasificación de clientes**

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS

WITH ordenes_cliente AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT SalesOrderID) AS amount_of_orders
    FROM workspace.electiva.orders
    WHERE YEAR(OrderDate) = 2011
    GROUP BY CustomerID
)

SELECT
    oc.amount_of_orders,
    pu.user_name,
    CONCAT(c.FirstName, ' ', c.LastName) AS customerfullname,

    CASE
        WHEN oc.amount_of_orders = 3 THEN 'Oro'
        WHEN oc.amount_of_orders = 2 THEN 'Plata'
        WHEN oc.amount_of_orders < 2 THEN 'Ocasional'
    END AS user_level,

    c.CustomerID AS customerid

FROM ordenes_cliente oc

INNER JOIN workspace.electiva.customers c
    ON oc.CustomerID = c.CustomerID

INNER JOIN workspace.electiva.page_user pu
    ON c.CustomerID = pu.customerid;

In [0]:
%sql
SELECT *
FROM workspace.electiva.customer_class
LIMIT 100;

Se creó una vista materializada denominada customer_class, mediante la cual se contabilizaron las órdenes distintas realizadas por cada cliente durante 2011. A partir de esta información, se establecieron tres categorías: Oro para quienes realizaron tres órdenes, Plata para quienes realizaron dos y Ocasional para quienes realizaron menos de dos.

Esta clasificación permite identificar los diferentes niveles de actividad de los clientes y proporciona información útil para diseñar campañas publicitarias y estrategias de fidelización orientadas a incrementar las ventas.

El desarrollo de los cuatro puntos permitió aplicar diferentes herramientas de SQL en Databricks para organizar, transformar, relacionar y analizar datos empresariales. Mediante la creación de tablas, consultas, vistas y vistas materializadas, fue posible obtener información útil sobre los clientes, sus compras y el comportamiento de las ventas. Estos procedimientos demuestran cómo la analítica de datos puede apoyar la toma de decisiones y la formulación de estrategias comerciales dentro de una empresa.